In [1]:
import numpy as np
import pandas as pd
pd.set_option("display.max_columns", None)

In [2]:
import seaborn as sns
import matplotlib.pyplot as plt

In [3]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import mean_squared_error

In [4]:
import pickle

In [5]:
import mlflow
mlflow.set_tracking_uri("sqlite:////workspaces/mlops-zoomcamp/02-experiment-tracking/mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/workspaces/mlops-zoomcamp/02-experiment-tracking/mlruns/1', creation_time=1781342302013, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1781342302013, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [6]:
import xgboost as xgb

"""
hyperopt is used to automatically search for good hyperparameters
    - fmin runs the optimization
    - tpe.suggest is the search algorithm
    - hp defines the search space
    - Trials() stores the results of each attempt
    - STATUS_OK tells Hyperopt the run completed successfully
"""
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/hyperopt/atpe.py:19: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


In [ ]:
# Optuna is a more popular choice today over hyperopt

In [7]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    
    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

In [9]:
df_train = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet')
df_val = read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet')

In [10]:
len(df_train), len(df_val)

(73908, 61921)

In [11]:
df_train['PU_DO'] = df_train['PULocationID'] + '_' + df_train['DOLocationID']
df_val['PU_DO'] = df_val['PULocationID'] + '_' + df_val['DOLocationID']

In [12]:
categorical = ['PU_DO'] #['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [13]:
target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [14]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

mse = mean_squared_error(y_val, y_pred)
rmse = np.sqrt(mse)
print(f'mse = {mse}')
print(f'rmse = {rmse}')

mse = 60.19766166037688
rmse = 7.758715206809494


In [15]:
with open('/workspaces/mlops-zoomcamp/models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [ ]:
with mlflow.start_run(): 

    mlflow.set_tag("developer", "chan")
    mlflow.log_param("train-data-path", "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet")
    mlflow.log_param("valid-data-path", "https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet")

    alpha = 0.01
    mlflow.log_param("alpha", alpha)

    lr = Lasso(alpha)
    X_train = X_train.astype("float32")
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="/workspaces/mlops-zoomcamp/models/lin_reg.bin", artifact_path='models_pickle')

/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/sklearn/linear_model/_coordinate_descent.py:784: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 3.316482e+05, tolerance: 9.882e+02
  model = cd_fast.sparse_enet_coordinate_descent(


Bad pipe message: %s [b'"Chromium";v="148", "Google Chrome";v="148", "Not/A)Brand']
Bad pipe message: %s [b'v="99"\r\nsec-ch-ua-mobile: ?0\r\nsec', b'h-ua-platform: "macOS"\r\nUpgrade-Insecure-Req', b'sts: 1\r\nUser-Agent: Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/14', b'0.0.0 Safari/537.36\r\nAccept: text/html,application/xhtm']
Bad pipe message: %s [b'xml,application/xml;q=0.9,image/avif,image/webp,image/apng,*/*;q=0.8,application/signed-exchange;v=b3;q=0.7', b'Sec-Fetch-Si']
Bad pipe message: %s [b': none\r\nSec-Fetch-Mode: navigate\r\nSec-Fetch-User: ?1\r\nSec-Fetch-Dest: document\r\nAccept-Encoding: gzip, deflate, br,', b'std\r\nAccept-Language: en-US,en;']
Bad pipe message: %s [b'0.9\r\nCookie: _xsrf=2|22415bc8|6f287070885ad552d1d3d12eb4250529|1780844093; username-127-0-0-1-8888="2|1:0|10:178', b'44123|23:username-127-0-0-1-8888|200:eyJ1c2Vybm']
Bad pipe message: %s [b'ZSI6ICI5ZWI2ZjA3NGVkNWY0ODBjOGFlOWFhOGRmNjk4NTExOSIsICJuYW1l

In [ ]:
# Convert data into XGBoost’s format - DMatrix is XGBoost’s optimized data format
train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

In [ ]:
# This function trains one XGBoost model using one set of hyperparameters

def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, 'validation')],
            early_stopping_rounds=50
        )
        y_pred = booster.predict(valid)
        # rmse = mean_squared_error(y_val, y_pred, squared=False)
        mse = mean_squared_error(y_val, y_pred)
        rmse = np.sqrt(mse)

        mlflow.log_metric("rmse", rmse)

    return {'loss': rmse, 'status': STATUS_OK}

In [ ]:
# # Define the hyperparameter search space
# search_space = {
#     'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
#     'learning_rate': hp.loguniform('learning_rate', -3, 0),
#     'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
#     'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
#     'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
#     'objective': 'reg:linear',
#     'seed': 42
# }

# # Run hyperparameter optimization
# best_result = fmin(
#     fn=objective,
#     space=search_space,
#     algo=tpe.suggest,
#     max_evals=50,
#     trials=Trials()
# )

In [ ]:
# A run is one experiment attempt. When you write: with mlflow.start_run(): you are saying to put all of these params, metrics, and artifacts under one MLflow run
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 30,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=100,#1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    #rmse = mean_squared_error(y_val, y_pred, squared=False)
    mse = mean_squared_error(y_val, y_pred)
    rmse = np.sqrt(mse)
    mlflow.log_metric("rmse", rmse)

    # First, this saves the preprocessor to your local project directory:
    with open("/workspaces/mlops-zoomcamp/models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    # Then this line uploads/logs that local file into MLflow artifacts:
    mlflow.log_artifact("/workspaces/mlops-zoomcamp/models/preprocessor.b", artifact_path="preprocessor")

    # For the model, you are not manually saving it as a pickle file first.
    # MLflow knows how to save XGBoost models because you are using the XGBoost-specific MLflow function: mlflow.xgboost.log_model(...)
    # So MLflow handles the model serialization for you.
    mlflow.xgboost.log_model(booster, name="models_mlflow")


/workspaces/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [16:15:20] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:11.44482
[1]	validation-rmse:10.77202
[2]	validation-rmse:10.18363
[3]	validation-rmse:9.67396
[4]	validation-rmse:9.23166
[5]	validation-rmse:8.84808
[6]	validation-rmse:8.51883
[7]	validation-rmse:8.23597
[8]	validation-rmse:7.99320
[9]	validation-rmse:7.78709
[10]	validation-rmse:7.61022
[11]	validation-rmse:7.45952
[12]	validation-rmse:7.33049
[13]	validation-rmse:7.22098
[14]	validation-rmse:7.12713
[15]	validation-rmse:7.04752
[16]	validation-rmse:6.98005
[17]	validation-rmse:6.92232
[18]	validation-rmse:6.87112
[19]	validation-rmse:6.82740
[20]	validation-rmse:6.78995
[21]	validation-rmse:6.75792
[22]	validation-rmse:6.72994
[23]	validation-rmse:6.70547
[24]	validation-rmse:6.68390
[25]	validation-rmse:6.66421
[26]	validation-rmse:6.64806
[27]	validation-rmse:6.63280
[28]	validation-rmse:6.61924
[29]	validation-rmse:6.60773
[30]	validation-rmse:6.59777
[31]	validation-rmse:6.58875
[32]	validation-rmse:6.58107
[33]	validation-rmse:6.57217
[34]	validation-rmse: